# Iso on a free Colab GPU

Use this when your own computer is too slow. Runtime → Change runtime type → **T4 GPU**, then run the cells top to bottom.

Your songs and results go to your Google Drive under `MyDrive/iso/`, so they survive the session.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get Iso

If the repository is private, add a GitHub token (repo read access) under the 🔑 *Secrets* tab as `GITHUB_TOKEN` first.

In [ ]:
import os
from google.colab import userdata
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
url = f"https://{token}@github.com/andrewphillipdoss/isolator.git" if token else "https://github.com/andrewphillipdoss/isolator.git"
!git clone -q --depth 1 --branch claude/drum-isolation-cleanup-q2lxxh {url} /content/isolator
!pip install -q -e "/content/isolator[all]"
!iso doctor

## 2. Songs from Google Drive

Put songs in `MyDrive/iso/in/`. Models are cached in `MyDrive/iso/models`, so they download once (~2 GB).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
base = '/content/drive/MyDrive/iso'
os.makedirs(f'{base}/in', exist_ok=True)
print(os.listdir(f'{base}/in'))

## 3. A trigger kit (optional, free)

Imports Hydrogen's GMRockKit (GPL). Skip it if you'll use the MIDI with your own drum sampler.

In [ ]:
!git clone -q --depth 1 --filter=blob:none --sparse https://github.com/hydrogen-music/hydrogen /content/hydrogen
!cd /content/hydrogen && git sparse-checkout set data/drumkits
!iso kit import /content/hydrogen/data/drumkits/GMRockKit --dest /content/kits/GMRockKit

## 4. Make the layer kits

`--preset max` is the slowest and best; on a T4 a 4-minute song takes a few minutes.

In [ ]:
!iso --models-dir {base}/models split {base}/in/* --preset max --kit /content/kits/GMRockKit -o {base}/out

Results are in `MyDrive/iso/out/<song>/`: stems, MIDI, `layer_kit.rpp`. Download the folder or open it from Drive on your computer.

## Optional: the browser mixer from Colab

This opens a temporary public link (Cloudflare quick tunnel). The link carries a random token, and the page refuses requests without it. Don't share the link.

In [ ]:
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
import subprocess, time, re, secrets
token = secrets.token_urlsafe(16)
srv = subprocess.Popen(['iso', '--models-dir', f'{base}/models', 'serve', '-o', f'{base}/out', '--port', '8765', '--token', token])
tun = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8765'], stderr=subprocess.PIPE, text=True)
for line in tun.stderr:
    m = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
    if m:
        print('Open:', f"{m.group(0)}/?token={token}"); break